<a href="https://colab.research.google.com/github/fabianxox/machinelearning/blob/main/naive_bayes_scratch_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
"""
main()
│
├── load_dataset()
│
├── train(df)
│      │
│      ├── count_classes()
│      ├── count_features()
│      └── calculate_probabilities()
│
├── predict(new_email)
│
└── evaluate()

"""

'\nmain()\n│\n├── load_dataset()\n│\n├── train(df)\n│      │\n│      ├── count_classes()\n│      ├── count_features()\n│      └── calculate_probabilities()\n│\n├── predict(new_email)\n│\n└── evaluate()\n\n'

In [49]:
import pandas as pd

In [50]:
def load_dataset():
  data = {
    "FREE":  ["Yes", "Yes", "No", "No", "Yes", "No"],
    "WIN":   ["No",  "No",  "No", "No", "No", "Yes"],
    "CLICK": ["Yes", "No",  "Yes", "No", "Yes", "No"],
    "Spam":  ["Yes", "Yes", "No", "No", "No", "No"]
    }
  table= pd.DataFrame(data)
  return table

In [51]:
data= load_dataset()
print(data)
#print(list(data.columns))

  FREE  WIN CLICK Spam
0  Yes   No   Yes  Yes
1  Yes   No    No  Yes
2   No   No   Yes   No
3   No   No    No   No
4  Yes   No   Yes   No
5   No  Yes    No   No


In [52]:
def train_table(data):
  counts= {}
  spam_counts= {
      "Yes": 0,
      "No": 0
  }
  for row in range(data.shape[0]):
    class_name= data["Spam"][row]
    spam_counts[class_name]+=1
    for feature in data.columns:
      if feature == "Spam":
        continue
      value= data[feature][row]
      if feature not in counts:
        counts[feature]= {}

      if class_name not in counts[feature]:
          counts[feature][class_name]= {}
          counts[feature][class_name]["Yes"]=0
          counts[feature][class_name]["No"]=0


      counts[feature][class_name][value]+=1


  return counts, spam_counts


In [53]:
counts, spam_counts= train_table(data)
print(counts)

print(spam_counts)


{'FREE': {'Yes': {'Yes': 2, 'No': 0}, 'No': {'Yes': 1, 'No': 3}}, 'WIN': {'Yes': {'Yes': 0, 'No': 2}, 'No': {'Yes': 1, 'No': 3}}, 'CLICK': {'Yes': {'Yes': 1, 'No': 1}, 'No': {'Yes': 2, 'No': 2}}}
{'Yes': 2, 'No': 4}


In [54]:
def cal_priors(spam_counts):
  total= sum(spam_counts.values())

  priors= {}
  for key, value in spam_counts.items():
    priors[key]= value/total

  return priors


In [55]:
def lap_smooth(count, total):
    return (count + 1) / (total + 2)

In [56]:
def cal_probs(counts):
  probs= {}
  for feature in counts:
    for class_name in counts[feature]:
      for value in counts[feature][class_name]:
        if feature not in probs:
          probs[feature]= {}
        if class_name not in probs[feature]:
          probs[feature][class_name]= {}
        if value not in probs[feature][class_name]:
          probs[feature][class_name][value]=0

        probs[feature][class_name][value]= lap_smooth(counts[feature][class_name][value], sum(counts[feature][class_name].values()))
  return probs

In [57]:
priors= cal_priors(spam_counts)
print(priors)

{'Yes': 0.3333333333333333, 'No': 0.6666666666666666}


In [66]:
probs= cal_probs(counts)
print(probs)

{'FREE': {'Yes': {'Yes': 0.75, 'No': 0.25}, 'No': {'Yes': 0.3333333333333333, 'No': 0.6666666666666666}}, 'WIN': {'Yes': {'Yes': 0.25, 'No': 0.75}, 'No': {'Yes': 0.3333333333333333, 'No': 0.6666666666666666}}, 'CLICK': {'Yes': {'Yes': 0.5, 'No': 0.5}, 'No': {'Yes': 0.5, 'No': 0.5}}}


In [59]:
def predict(mail, probs, priors):
  scores = priors.copy()
  for words, word_value in mail.items():
    for class_name in scores.keys():
      scores[class_name]*= probs[words][class_name][word_value]

  high_p= -1
  for key, value in scores.items():
    if value>high_p:
      high_p= value
      category= key

  return category, high_p

In [60]:
new_email = {
    "FREE": "Yes",
    "WIN": "No",
    "CLICK": "No"
}

cat, p= predict(new_email, probs, priors)
print(cat)
print(p)

Yes
0.09375


In [64]:
def evaluate(data, priors, probs):
  crct= 0
  for row in range(data.shape[0]):
    email= {}
    for feature in data.columns:
      if feature == "Spam":
        continue
      email[feature]= data[feature][row]
    actual= data["Spam"][row]
    category, p= predict(email, probs, priors)
    if(category == actual):
      crct+=1
    print(f"for row: {row+1}, the predicted value is: {category}, {p} , the actual value is: {actual}")

  return crct/data.shape[0]

In [65]:

eval= evaluate(data, priors, probs)
print(eval)

for row: 1, the predicted value is: Yes, 0.09375 , the actual value is: Yes
for row: 2, the predicted value is: Yes, 0.09375 , the actual value is: Yes
for row: 3, the predicted value is: No, 0.14814814814814814 , the actual value is: No
for row: 4, the predicted value is: No, 0.14814814814814814 , the actual value is: No
for row: 5, the predicted value is: Yes, 0.09375 , the actual value is: No
for row: 6, the predicted value is: No, 0.07407407407407407 , the actual value is: No
0.8333333333333334
